# Exploring NYC Taxi Fare Data with PyCanopy

PyCanopy is a spatial query layer for Polars. It wraps a Polars DataFrame with an adaptive spatial index (KD-tree, R-tree, or grid) and exposes a lazy, declarative API whose cost-model-driven planner decides at runtime how to execute each query.

This notebook walks through PyCanopy's core operations using **3.5 million NYC Yellow Taxi trips** from the Kaggle Taxi Fare dataset, which includes GPS coordinates for pickup and dropoff, fare amount, and passenger count.

We will cover:

- Building a `SpatialFrame`, choosing an index mode, and a coordinate system
- Inspecting the query plan with `.explain()`
- Bounding-box filters with `.range_query()`
- Combining scalar and spatial predicates
- Single-point k-nearest neighbours with `.knn()`
- Proximity joins in real metres with `.within_distance_join()`
- Aggregating over a join with `.group_by().agg()`
- Matching every query point to its k nearest neighbours with `.knn_join()`
- Running several queries from a shared plan prefix with `collect_all()`

In [ ]:
!pip install pycanopy "polars==1.41.2" -q

In [ ]:
import polars as pl
import pycanopy as pc
from pycanopy import SpatialFrame
from pycanopy.lazy import SpatialLazyFrame

## Loading the Data

The dataset is the NYC Yellow Taxi Fare prediction dataset, hosted publicly on Hugging Face. We load one Parquet shard (~255 MB, ~3.5 M rows). We filter to the valid NYC bounding box straight after loading.

In [ ]:
!wget -q "https://huggingface.co/datasets/TaherMAfini/taxi_dataset/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet" -O /tmp/taxi.parquet

raw = pl.read_parquet("/tmp/taxi.parquet")
print(f"{raw.height:,} rows  |  columns: {raw.columns}")

In [ ]:
NYC = {"lon_min": -74.05, "lon_max": -73.70, "lat_min": 40.48, "lat_max": 40.92}

trips = raw.filter(
    pl.col("pickup_longitude").is_between(NYC["lon_min"], NYC["lon_max"])
    & pl.col("pickup_latitude").is_between(NYC["lat_min"], NYC["lat_max"])
    & pl.col("fare_amount").gt(0)
).select(
    [
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
        "fare_amount",
    ]
)

print(f"{trips.height:,} trips with valid NYC coordinates")
trips.head(4)

## Setting Up a SpatialFrame

A `SpatialFrame` wraps a Polars DataFrame with a spatial index and is the entry point for all query planning. You provide the column names for x (longitude) and y (latitude) coordinates, and optionally set an `index_mode` and a `coordinate_system`.

| Mode | Behaviour |
|------|-----------|
| `"auto"` | Default. Builds the index only when the cost model determines it beats a full scan for the given probe. |
| `"eager"` | Builds the index immediately at construction time. No build latency on the first query. |
| `"none"` | Always brute-force scans. Useful as a baseline or for very small frames. |

The `coordinate_system` controls how distances are measured. The default `"planar"` is plain Euclidean distance in the coordinate units, which would be degrees here. Since our columns are real lon/lat, we pass `"geographic"` instead, so PyCanopy reads them as WGS84 degrees and measures great-circle (haversine) distance in **metres**. Distance-threshold operations like `.within_distance_join()` honour this. kNN stays a nearest-neighbour search ranked in degrees, which is equivalent locally.

We use `index_mode="eager"` so the index is ready before any query runs, and `coordinate_system="geographic"` so proximity thresholds are in metres.

In [ ]:
sf = SpatialFrame(
    trips,
    x_col="pickup_longitude",
    y_col="pickup_latitude",
    index_mode="eager",
    coordinate_system="geographic",
)
print(f"SpatialFrame ready — {trips.height:,} pickup points indexed")

## Inspecting the Query Plan

Every call on a `SpatialLazyFrame` appends a plan node without executing anything. `.explain()` runs the optimizer on the current plan and shows what will actually execute, including any reordering of predicates.

Plans use Polars' `FROM`-chain convention: the bottom node runs first and the top is the final result.

Below we declare a scalar filter after a range query, then inspect what the optimizer produces.

In [ ]:
plan = (
    sf.lazy()
    .range_query(min_x=-74.02, min_y=40.70, max_x=-73.93, max_y=40.82)
    .filter(pl.col("passenger_count") == 1)
)
print(plan.explain())

The optimizer moved the scalar filter below the range query. The `passenger_count` filter runs first on all rows, reducing the candidate set before the spatial index is probed. Declaration order does not constrain execution order.

## Filtering by Location

`.range_query(min_x, min_y, max_x, max_y)` keeps only the rows whose coordinates fall inside the bounding box. Here we pull every pickup that started in the Manhattan core.

In [ ]:
manhattan = sf.lazy().range_query(min_x=-74.02, min_y=40.70, max_x=-73.93, max_y=40.82).collect()
print(f"{manhattan.height:,} pickups in the Manhattan core")
manhattan.select(["pickup_datetime", "pickup_longitude", "pickup_latitude", "fare_amount"]).head(5)

### Combining Spatial and Scalar Filters

Scalar and spatial predicates compose freely. The optimizer always places scalar filters ahead of spatial ones regardless of declaration order, so you can write the query in whichever order reads most naturally.

In [ ]:
expensive_manhattan = (
    sf.lazy()
    .filter(pl.col("fare_amount") > 20)
    .filter(pl.col("passenger_count") == 1)
    .range_query(min_x=-74.02, min_y=40.70, max_x=-73.93, max_y=40.82)
    .collect()
)
print(f"{expensive_manhattan.height:,} solo pickups with fare over $20 in the Manhattan core")
expensive_manhattan.select(
    ["pickup_datetime", "pickup_longitude", "pickup_latitude", "fare_amount"]
).head(5)

## Finding Nearby Trips with kNN

`.knn(x, y, k)` returns the `k` rows nearest to a single query point. It is useful for questions like "what trips started closest to this corner?"

Here we find the 15 pickups closest to Grand Central Terminal.

In [ ]:
GRAND_CENTRAL = (-73.9772, 40.7527)

near_grand_central = sf.lazy().knn(x=GRAND_CENTRAL[0], y=GRAND_CENTRAL[1], k=15).collect()
near_grand_central.select(["pickup_datetime", "pickup_longitude", "pickup_latitude", "fare_amount"])

### kNN with a Scalar Pre-filter

A `.filter()` before `.knn()` restricts the candidate pool before the spatial search runs. Below we find the 10 pickups nearest to Penn Station that carried more than one passenger.

In [ ]:
PENN_STATION = (-73.9934, 40.7506)

near_penn_groups = (
    sf.lazy()
    .filter(pl.col("passenger_count") > 1)
    .knn(x=PENN_STATION[0], y=PENN_STATION[1], k=10)
    .collect()
)
near_penn_groups.select(
    ["pickup_datetime", "pickup_longitude", "pickup_latitude", "passenger_count", "fare_amount"]
)

## Proximity Joins: Pickups Near Landmarks

`.within_distance_join(query_df, x_col, y_col, distance)` pairs every point in `query_df` with all dataset points within `distance` of it. Because the frame is geographic, `distance` is in **metres**, measured as a great-circle arc. The result concatenates columns from both sides for every matched pair.

We build a small DataFrame of NYC landmarks and find every pickup in the dataset within 550 metres of each one.

In [ ]:
landmarks = pl.DataFrame(
    {
        "landmark": [
            "Times Square",
            "Grand Central",
            "Penn Station",
            "JFK Airport",
            "Central Park S",
        ],
        "lon": [-73.9855, -73.9772, -73.9934, -73.7789, -73.9765],
        "lat": [40.7580, 40.7527, 40.7506, 40.6413, 40.7665],
    }
)

# metres, since the frame uses coordinate_system="geographic"
RADIUS = 550

near_landmarks = (
    sf.lazy().within_distance_join(landmarks, x_col="lon", y_col="lat", distance=RADIUS).collect()
)
print(f"{near_landmarks.height:,} pickup-landmark pairs")

near_landmarks.group_by("landmark").agg(pl.len().alias("pickups")).sort("pickups", descending=True)

## Aggregating Over a Join Without Materialising the Pair Frame

When you only need summary statistics from a join, chaining `.group_by().agg()` directly after the spatial join tells PyCanopy to fold each probe morsel into per-group partials as it streams. The full pair DataFrame is never held in memory.

Here we compute pickup counts and average fares per landmark in one pass.

In [ ]:
landmark_stats = (
    sf.lazy()
    .within_distance_join(landmarks, x_col="lon", y_col="lat", distance=RADIUS)
    .group_by(["landmark"])
    .agg(
        pickup_count=pc.agg.count(),
        avg_fare=pc.agg.mean("fare_amount"),
        avg_passengers=pc.agg.mean("passenger_count"),
    )
)
landmark_stats.sort("pickup_count", descending=True)

## kNN Join: Nearest Pickups for Every Dropoff

`.knn_join(query_df, x_col, y_col, k)` runs kNN at scale: for every row in `query_df` it finds the `k` nearest rows in the `SpatialFrame`. Result columns are the query side followed by the matched side.

Large probes are streamed in morsels automatically so peak memory stays bounded.

Here we ask: for a sample of 50,000 dropoff locations, which 3 pickup locations in the dataset were nearest? This gives a sense of where supply was concentrated relative to where passengers were left.

In [ ]:
dropoffs = (
    trips.filter(
        pl.col("dropoff_longitude").is_between(NYC["lon_min"], NYC["lon_max"])
        & pl.col("dropoff_latitude").is_between(NYC["lat_min"], NYC["lat_max"])
    )
    .select(["dropoff_longitude", "dropoff_latitude", "fare_amount"])
    .sample(n=50_000, seed=42)
)

nearest_pickups = (
    sf.lazy()
    .knn_join(dropoffs, x_col="dropoff_longitude", y_col="dropoff_latitude", k=3)
    .select(
        [
            "dropoff_longitude",
            "dropoff_latitude",
            "pickup_longitude",
            "pickup_latitude",
            "fare_amount",
        ]
    )
    .collect()
)
print(f"{nearest_pickups.height:,} pairs  ({dropoffs.height:,} dropoffs x k=3 nearest pickups)")
nearest_pickups.head(6)

## Multiple Queries from a Shared Plan Prefix

`SpatialLazyFrame.collect_all(frames)` detects the common plan prefix shared by a list of lazy frames, executes it once with a Polars `.cache()`, and fans the cached result out to each branch. Any scalar filters in the shared prefix run exactly once regardless of how many branches follow.

Below we compare solo-passenger pickup volumes and average fares across three Manhattan neighbourhoods. All three plans share the `passenger_count == 1` filter, so it runs once.

In [ ]:
solo = sf.lazy().filter(pl.col("passenger_count") == 1)

midtown_lf = solo.range_query(min_x=-74.00, min_y=40.74, max_x=-73.95, max_y=40.77)
downtown_lf = solo.range_query(min_x=-74.02, min_y=40.70, max_x=-73.97, max_y=40.72)
harlem_lf = solo.range_query(min_x=-73.96, min_y=40.80, max_x=-73.92, max_y=40.83)

midtown, downtown, harlem = SpatialLazyFrame.collect_all([midtown_lf, downtown_lf, harlem_lf])

pl.concat(
    [
        midtown.with_columns(pl.lit("Midtown").alias("neighbourhood")),
        downtown.with_columns(pl.lit("Downtown").alias("neighbourhood")),
        harlem.with_columns(pl.lit("Harlem").alias("neighbourhood")),
    ]
).group_by("neighbourhood").agg(
    pl.len().alias("solo_pickups"),
    pl.col("fare_amount").mean().round(2).alias("avg_fare"),
    pl.col("fare_amount").max().round(2).alias("max_fare"),
).sort("solo_pickups", descending=True)